In [77]:
docs_name = [
    'Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt',
    'Datasets/Unstructured_Data/3okobat/3okobat_num36_2020.txt',
    'Datasets/Unstructured_Data/3okobat/3okobat.pdf',
    'Datasets/Unstructured_Data/Asle7a/Asle7a.pdf',
    'Datasets/Unstructured_Data/child/child.pdf',
    'Datasets/Unstructured_Data/dostor_gena2y/dostor.pdf',
    'Datasets/Unstructured_Data/drugs/drugs.pdf',
    'Datasets/Unstructured_Data/egra2at_gena2ya/egra2at_gena2ya.pdf',
    'Datasets/Unstructured_Data/erhab/erhab.txt',
    'Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf',
    'Datasets/Unstructured_Data/technology/tech.txt'
]

In [78]:
import fitz  # PyMuPDF
import os
from langchain_classic.schema import Document

base_url = "/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject"

def load_pdf_pymupdf(pdf_path):
    """Load PDF content using PyMuPDF"""
    try:
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()
        return text
    except Exception as e:
        print(f"Error loading PDF {pdf_path}: {e}")
        return None

def load_text_file(txt_path):
    """Load text file content"""
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        print(f"Error loading text file {txt_path}: {e}")
        return None

# Load all documents as LangChain Document objects
documents = []

for doc_name in docs_name:
    full_path = os.path.join(base_url, doc_name)
    ext = os.path.splitext(doc_name)[1].lower()

    content = None
    if ext == ".pdf":
        content = load_pdf_pymupdf(full_path)
    elif ext == ".txt":
        content = load_text_file(full_path)
    else:
        print(f"⚠ Skipping unsupported file type: {doc_name}")

    if content:
        documents.append(
            Document(
                page_content=content,
                metadata={"source": doc_name}  # Keep track of the file name
            )
        )
        print(f"✓ Successfully loaded: {doc_name}")
    else:
        print(f"✗ Failed to load: {doc_name}")

print(f"\nTotal loaded: {len(documents)}/{len(docs_name)} documents")


✓ Successfully loaded: Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt
✓ Successfully loaded: Datasets/Unstructured_Data/3okobat/3okobat_num36_2020.txt
✓ Successfully loaded: Datasets/Unstructured_Data/3okobat/3okobat.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/Asle7a/Asle7a.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/child/child.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/dostor_gena2y/dostor.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/drugs/drugs.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/egra2at_gena2ya/egra2at_gena2ya.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/erhab/erhab.txt
✓ Successfully loaded: Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/technology/tech.txt

Total loaded: 11/11 documents


In [79]:
documents

[Document(metadata={'source': 'Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt'}, page_content='المادة الأولى: يستبدل بنص المادة ٣٩٢ من قانون العقوبات الصادر بالقانون رقم ٥٨ لسنة 1937 النص الآتي: مادة ٣٩٢ كل من صدر عليه حكم قضائي واجب النفاذ بدفع نفقة لزوجه أو أقاربه أو أصهاره أو أجرة حضانة أو رضاعة أو مسكن وامتنع عن الدفع مع قدرته عليه لمدة ثلاثة أشهر بعد التنبيه عليه بالدفع، يعاقب بالحبس مدة لا تزيد على سنة وبغرامة لا تتجاوز خمسة آلاف جنيه أو بإحدى هاتين العقوبتين، ولا ترفع الدعوى عليه إلا بناء على شكوى أو طلب من صاحب الشأن، وإذا رفعت بعد الحكم عليه دعوى ثانية عن هذه الجريمة فتكون عقوبته الحبس مدة لا تزيد على سنة، ويترتب على الحكم الصادر بالإدانة تعليق استفادة المحكوم عليه من الخدمات المطلوب الحصول عليها بمناسبة ممارسته نشاطه المهني والتي تقدمها الجهات الحكومية والهيئات العامة ووحدات القطاع العام وقطاع الأعمال العام والجهات التي تؤدي خدمات مرافق عامة، حتى أداء ما تجمد في ذمته لصالح المحكوم له وبنك ناصر الاجتماعي حسب الأحوال.\nالمادة الثانية: ينشر هذا القانون في الجريدة الرسم

In [80]:
import re
from langchain_classic.schema import Document

def advanced_legal_split_langchain(text, source_name):
    
    documents = []
    
    patterns = {
    'law': r'قانون\s*رقم\s*\d+\s*لسنة\s*\d+',
    'book': r'(?:الكتاب|كتاب)\s*(?:الأول|الثاني|الثالث|الرابع|\d+)',
    'section': r'(?:الباب|باب)\s*(?:الأول|الثاني|الثالث|الرابع|\d+)',
    'chapter': r'(?:الفصل|فصل)\s*(?:الأول|الثاني|الثالث|الرابع|\d+)',
    'article': r'\(?\s*(?:المادة|مادة)[\s\(:\-]*([0-9٠-٩]+|الأول|الثاني|الثالث|الرابع|الخامس|السادس|السابع|الثامن|التاسع|العاشر)\s*\)?'
}


    
    article_pattern = patterns['article']
    matches = list(re.finditer(article_pattern, text))
    
    current_law = ""
    current_section = ""
    current_chapter = ""
    
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        
        article_text = text[start:end].strip()
        article_number_match = re.search(r'\d+', match.group())
        article_number = article_number_match.group() if article_number_match else "غير محدد"
        
        context_text = text[max(0, start-500):start]
        
        law_match = re.search(patterns['law'], context_text)
        if law_match:
            current_law = law_match.group()
        
        section_match = re.search(patterns['section'], context_text)
        if section_match:
            current_section = section_match.group()
        
        chapter_match = re.search(patterns['chapter'], context_text)
        if chapter_match:
            current_chapter = chapter_match.group()
        
        doc = Document(
            page_content=article_text,
            metadata={
                'source': source_name,
                'article_number': article_number,
                'law': current_law,
                'section': current_section,
                'chapter': current_chapter,
                'type': 'article'
            }
        )
        documents.append(doc)
    
    return documents

In [94]:
all_legal_docs = []

In [95]:
for doc in documents:
    source = doc.metadata.get('source', 'unknown')
    content = doc.page_content
    
    docs = advanced_legal_split_langchain(content, source)
    all_legal_docs.extend(docs)
    print(f"{source}: {len(docs)} articles extracted")

print(f"\nTotal articles: {len(all_legal_docs)}")

if all_legal_docs:
    print("\nمثال على مادة:")
    print(f"المصدر: {all_legal_docs[1].metadata['source']}")
    print(f"رقم المادة: {all_legal_docs[1].metadata['article_number']}")
    print(f"القانون: {all_legal_docs[1].metadata['law']}")
    print(f"الباب: {all_legal_docs[1].metadata['section']}")
    print(f"المحتوى: {all_legal_docs[1].page_content}...")

Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt: 4 articles extracted
Datasets/Unstructured_Data/3okobat/3okobat_num36_2020.txt: 3 articles extracted
Datasets/Unstructured_Data/3okobat/3okobat.pdf: 513 articles extracted
Datasets/Unstructured_Data/Asle7a/Asle7a.pdf: 49 articles extracted
Datasets/Unstructured_Data/child/child.pdf: 40 articles extracted
Datasets/Unstructured_Data/dostor_gena2y/dostor.pdf: 166 articles extracted
Datasets/Unstructured_Data/drugs/drugs.pdf: 25 articles extracted
Datasets/Unstructured_Data/egra2at_gena2ya/egra2at_gena2ya.pdf: 61 articles extracted
Datasets/Unstructured_Data/erhab/erhab.txt: 67 articles extracted
Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf: 334 articles extracted
Datasets/Unstructured_Data/technology/tech.txt: 53 articles extracted

Total articles: 1315

مثال على مادة:
المصدر: Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt
رقم المادة: ٣٩٢
القانون: 
الباب: 
المحتوى: المادة ٣٩٢ من قانون العقوبات الصادر بالقانون 

In [97]:
file_path = "all_legal_articles.txt"

with open(file_path, "w", encoding="utf-8") as f:
    for i in range(0, 1315):
        article = all_legal_docs[i]
        f.write(f"المصدر: {article.metadata.get('source','')}\n")
        f.write(f"رقم المادة: {article.metadata.get('article_number','')}\n")
        f.write(f"القانون: {article.metadata.get('law','')}\n")
        f.write(f"الباب: {article.metadata.get('section','')}\n")
        f.write(f"المحتوى:\n{article.page_content}\n")
        f.write("*" * 50 + "\n\n")

print(f"تم حفظ جميع المواد في الملف: {file_path}")


تم حفظ جميع المواد في الملف: all_legal_articles.txt
